In [20]:
#!pip install pytorch-forecasting pytorch-lightning

In [38]:
import pandas as pd
import numpy as np
import pytorch_lightning as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

df = pd.read_parquet("sales_features.parquet")


In [39]:

df['is_holiday'] = df['is_holiday'].astype(bool).astype(str)
df['is_weekend'] = df['is_weekend'].astype(bool).astype(str)
df["day_of_week"] = df['day_of_week'].astype(str)
df["quarter"] = df['quarter'].astype(str)
 

# Convert zipcode to string and handle categoricals
df['zipcode'] = df['zipcode'].astype(int).astype(str).str.zfill(5)

# Encode static categorical variables
# for col in ['city', 'county', 'zipcode']:
#     df[col] = df[col].astype('category').cat.codes + 1

# Create unique group_id for each store
df['group_id'] = df.groupby(['name', 'address', 'zipcode']).ngroup().astype(str)

# Create time_idx (days since first observation per group)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['group_id', 'date'])
df['time_idx'] = df.groupby('group_id')['date'].transform(lambda x: (x - x.min()).dt.days)

print(df[['group_id', 'time_idx']].duplicated())


store
5876    False
5876    False
5876    False
5876    False
5876    False
        ...  
4317    False
4317    False
4317    False
4317    False
4317    False
Length: 254481, dtype: bool


In [40]:
# 2. Define aggregation rules
agg_rules = {
    # Static features (take first)
    'name': 'first',
    'time_idx': 'first',
    'address': 'first',
    'city': 'first',
    'zipcode': 'first',
    'county': 'first',
    'lon': 'first',
    'lat': 'first',
    'store_size': 'first',
    
    # Time-varying features
    'sale_dollars': 'sum',
    'sale_bottles': 'sum',
    'transaction_count': 'sum',
    'unique_transactions': 'sum',
    'sale_liters': 'sum',
    'sale_gallons': 'sum',
    'profit_margin': 'mean',
    'discount_factor': 'mean',
    'avg_price_per_bottle': 'mean',
    'avg_price_per_liter': 'mean',
    'avg_items_per_transaction': 'mean',
    'day_of_week': 'first',
    'month': 'first',
    'quarter': 'first',
    'year': 'first',
    'is_weekend': 'first',
    'day_of_month': 'first',
    'week_of_year': 'first',
    
    # Statistical features (recalculate after aggregation)
    'sale_dollars_mean': 'mean',
    'sale_dollars_std': 'std',
    'sale_dollars_min': 'min',
    'sale_dollars_max': 'max',
}


In [28]:
# 3. Aggregate duplicates
df_agg = df.groupby(['group_id', 'date']).agg(agg_rules).reset_index()

In [29]:
# 4. Recreate temporal features after aggregation
def recreate_features(group):
    group = group.sort_values('date')
    
    # Lag features
    for lag in [1, 7, 14, 30, 60]:
        group[f'sale_dollars_lag_{lag}d'] = group['sale_dollars'].shift(lag)
    
    # Rolling features
    for window in [7, 14, 30, 60, 90]:
        group[f'sale_dollars_rolling_mean_{window}D'] = (
            group['sale_dollars']
            .rolling(window=window, min_periods=1)
            .mean()
        )
        group[f'sale_dollars_rolling_std_{window}D'] = (
            group['sale_dollars']
            .rolling(window=window, min_periods=1)
            .std()
        )
    
    # Time features
    group['month_sin'] = np.sin(2 * np.pi * group['month'] / 12)
    group['month_cos'] = np.cos(2 * np.pi * group['month'] / 12)
    group['day_of_month_sin'] = np.sin(2 * np.pi * group['day_of_month'] / 31)
    group['day_of_month_cos'] = np.cos(2 * np.pi * group['day_of_month'] / 31)
    
    return group

df_agg = df_agg.groupby('group_id').apply(recreate_features).reset_index(drop=True)

In [30]:
df_agg

,group_id,date,name,time_idx,address,city,zipcode,county,lon,lat,...,sale_dollars_rolling_mean_30D,sale_dollars_rolling_std_30D,sale_dollars_rolling_mean_60D,sale_dollars_rolling_std_60D,sale_dollars_rolling_mean_90D,sale_dollars_rolling_std_90D,month_sin,month_cos,day_of_month_sin,day_of_month_cos
0,0,2021-04-07,'DA BOOZE BARN / WEST BEND,0,108 S BROADWAY,WEST BEND,50597,PALO ALTO,-73.982421,40.305231,...,147.240,NaN,147.240000,NaN,147.240000,NaN,8.660254e-01,-0.500000,9.884683e-01,0.151428
1,0,2021-04-15,'DA BOOZE BARN / WEST BEND,8,108 S BROADWAY,WEST BEND,50597,PALO ALTO,-73.982421,40.305231,...,176.025,40.708137,176.025000,40.708137,176.025000,40.708137,8.660254e-01,-0.500000,1.011683e-01,-0.994869
2,0,2021-04-21,'DA BOOZE BARN / WEST BEND,14,108 S BROADWAY,WEST BEND,50597,PALO ALTO,-73.982421,40.305231,...,138.110,71.702287,138.110000,71.702287,138.110000,71.702287,8.660254e-01,-0.500000,-8.978045e-01,-0.440394
3,0,2021-04-28,'DA BOOZE BARN / WEST BEND,21,108 S BROADWAY,WEST BEND,50597,PALO ALTO,-73.982421,40.305231,...,175.860,95.539147,175.860000,95.539147,175.860000,95.539147,8.660254e-01,-0.500000,-5.712682e-01,0.820763
4,0,2021-05-05,'DA BOOZE BARN / WEST BEND,28,108 S BROADWAY,WEST BEND,50597,PALO ALTO,-73.982421,40.305231,...,156.444,93.438224,156.444000,93.438224,156.444000,93.438224,5.000000e-01,-0.866025,8.486443e-01,0.528964
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254476,999,2024-12-28,FAREWAY STORES #941 / GREENFIELD,1366,212 SW KENT,GREENFIELD,50849,ADAIR,-94.462719,41.303167,...,307.932,243.925683,329.262667,232.262269,340.261778,223.369051,-2.449294e-16,1.000000,-5.712682e-01,0.820763
254477,999,2025-01-10,FAREWAY STORES #941 / GREENFIELD,1379,212 SW KENT,GREENFIELD,50849,ADAIR,-94.462719,41.303167,...,302.522,246.728373,327.259667,233.721852,338.613778,224.589721,5.000000e-01,0.866025,8.978045e-01,-0.440394
254478,999,2025-01-19,FAREWAY STORES #941 / GREENFIELD,1388,212 SW KENT,GREENFIELD,50849,ADAIR,-94.462719,41.303167,...,328.653,258.117175,337.642833,241.277929,345.660444,229.444437,5.000000e-01,0.866025,-6.513725e-01,-0.758758
254479,999,2025-01-24,FAREWAY STORES #941 / GREENFIELD,1393,212 SW KENT,GREENFIELD,50849,ADAIR,-94.462719,41.303167,...,322.531,260.565568,336.383833,242.155847,345.012444,229.965559,5.000000e-01,0.866025,-9.884683e-01,0.151428


In [62]:
#!pip install holidays

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.9/917.9 kB 4.2 MB/s eta 0:00:0000:0100:01


In [31]:
from pandas.tseries.holiday import USFederalHolidayCalendar
import holidays

# 5. Add holiday features (using US holidays example)
us_holidays = holidays.US(years=df_agg['date'].dt.year.unique())
df_agg['is_holiday'] = df_agg['date'].isin(us_holidays).astype(int)
df_agg['holiday_name'] = df_agg['date'].map(us_holidays.get)

In [32]:
assert df_agg[['group_id', 'date']].duplicated().sum() == 0, "Duplicates still exist!"

In [36]:

# Static variables
static_categoricals = ['city', 'county', 'zipcode']
static_reals = ['lon', 'lat', 'store_size', 'store_avg_sales', 'city_avg_sales', 'county_avg_sales']

# Time-varying known variables (known in advance)
time_varying_known_categoricals = ['is_holiday', 'is_weekend', 'day_of_week', 'quarter']
time_varying_known_reals = ['month_sin', 'month_cos', 'day_of_month_sin', 'day_of_month_cos']

# Time-varying unknown variables (historical)
time_varying_unknown_reals = [
'sale_dollars',
'sale_dollars_lag_1d',
'sale_dollars_rolling_mean_7D',
'sale_dollars_rolling_std_7D',
# Include other lagged/rolling features
'transaction_count', 'unique_transactions', 'profit_margin'
]


In [37]:

max_encoder_length = 365
max_prediction_length = 14


# Split data temporally
train_cutoff = df['time_idx'].max() - max_prediction_length * 2
training = TimeSeriesDataSet(
    df[df.time_idx <= train_cutoff],
    time_idx="time_idx",
    target="sale_dollars",
    group_ids=["group_id", "time_idx"],
    static_categoricals=[],
    static_reals=static_reals,
    time_varying_known_categoricals=time_varying_known_categoricals,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_categoricals=static_categoricals,
    time_varying_unknown_reals=time_varying_unknown_reals,
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)



validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)


AssertionError: data index has to be unique